<a href="https://colab.research.google.com/github/2masCarvalho/Battleship/blob/main/parte1_CVD_v2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descoberta e Extração de Conhecimento de Dados
## Projeto Final — Parte 1
### Risco de Doenças Cardiovasculares (BRFSS 2021)

**Grupo:** [15]  
**Elementos:**
- Francisco Rainho 1 — nº 106312
- Manuel Pina 2 — nº 111131
- Pedro Vieira 3 — nº 111158
- Tomás Silva 4 — nº 111393

**Data de entrega:** 24 de abril de 2026

---
## 0. Bibliotecas e Configurações

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
SEED = 42
np.random.seed(SEED)
print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

### Configurar Caminho do Ficheiro CSV

Defina o caminho para o seu ficheiro `CVD_cleaned.csv` no Google Drive. Cada colaborador deverá ajustar esta variável para o seu próprio caminho.

In [ ]:
# Por favor, atualiza esta variável com o caminho correto para o teu ficheiro CVD_cleaned.csv.
# Exemplo: '/content/drive/MyDrive/PastaDoTeuTrabalho/CVD_cleaned.csv'
CSV_FILE_PATH = '/content/drive/MyDrive/CVD_cleaned.csv' # Caminho padrão

print(f'Caminho do CSV configurado para: {CSV_FILE_PATH}')

### Carregar o dataset

Para que o Colab possa aceder ao ficheiro `CVD_cleaned.csv`, precisas de o carregar para o teu Google Drive.

**Passos para carregar o ficheiro CSV para o Google Drive:**
1.  No painel esquerdo do Colab, clica no ícone de pasta (`Files`).
2.  Clica no ícone do Google Drive (o triângulo).
3.  Navega para a pasta `MyDrive` e depois para a pasta onde queres guardar o ficheiro (podes criar uma nova pasta, por exemplo, `Trabalho de DECD`).
4.  Clica no botão `Upload` (seta para cima) e seleciona o ficheiro `CVD_cleaned.csv` do teu computador.

Uma vez que o ficheiro esteja no Google Drive, o caminho para ele será algo como `/content/drive/MyDrive/caminho_da_tua_pasta/CVD_cleaned.csv`. Certifica-te de que o caminho no código abaixo corresponde à localização onde guardaste o ficheiro.

In [ ]:
df = pd.read_csv(CSV_FILE_PATH)
print(f'Dimensões: {df.shape[0]:,} registos × {df.shape[1]} variáveis')
display(df.head())

---
# 1. Compreensão do Problema *(Business Understanding)*

## 1.1 Contexto

As doenças cardiovasculares (DCV) são a principal causa de morte a nível mundial, responsáveis por cerca de 17,9 milhões de mortes por ano, segundo a Organização Mundial de Saúde. A identificação precoce de fatores de risco e de perfis populacionais vulneráveis é essencial para a prevenção e para a alocação eficiente de recursos de saúde pública.

O dataset utilizado provém do **Behavioral Risk Factor Surveillance System (BRFSS) 2021**, um sistema de vigilância do Center for Disease Control and Prevention (CDC) que recolhe dados de saúde de residentes dos EUA através de inquéritos telefónicos. O subconjunto utilizado contém **308.854 registos** e **19 variáveis** relacionadas com comportamentos de risco, condições de saúde crónicas e dados sociodemográficos.

## 1.2 Objetivos e Perguntas de Investigação

Este projeto visa explorar o risco de doenças cardiovasculares através de abordagens supervisionadas e não supervisionadas. Para a **Parte 1**, definimos as seguintes perguntas de investigação:

| # | Pergunta |
|---|---|
| P1 | Quais são as variáveis mais correlacionadas com a presença de doença cardíaca? |
| P2 | Existe uma relação entre o IMC (BMI), a diabetes e a doença cardíaca? |
| P3 | A prática de exercício físico e a qualidade da alimentação estão associadas a menor prevalência de DCV? |
| P4 | Existem perfis populacionais (clusters) distintos com diferentes níveis de risco cardiovascular? |

## 1.3 Variáveis Alvo

- **Target principal (classificação):** `Heart_Disease` — presença de doença coronária ou enfarte do miocárdio.
- **Target secundário (classificação):** `Diabetes` — diagnóstico de diabetes.
- **Perspetiva não supervisionada:** identificação de **perfis de saúde** (clusters) que permitam caracterizar grupos populacionais com padrões de risco semelhantes.

---
# 2. Compreensão dos Dados *(Data Understanding)*

## 2.1 Carregamento e Visão Geral

In [ ]:
df = pd.read_csv(CSV_FILE_PATH)
print(f'Dimensões: {df.shape[0]:,} registos × {df.shape[1]} variáveis')
df.head()

In [ ]:
# Tipos, valores nulos e cardinalidade
resumo = pd.DataFrame({
    'Tipo'      : df.dtypes,
    'Não-nulos' : df.notna().sum(),
    'Nulos'     : df.isna().sum(),
    '% Nulos'   : (df.isna().sum() / len(df) * 100).round(2),
    'Únicos'    : df.nunique(),
    'Exemplo'   : [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    'Valores Únicos': [df[c].dropna().unique().tolist() if df[c].nunique() <= 10 else str(df[c].dropna().unique()[:5].tolist()) + '...' for c in df.columns]
})
resumo

## 2.2 Descrição das Variáveis

| Variável | Tipo CRISP-DM | Descrição |
|---|---|---|
| General_Health | Ordinal | Estado de saúde auto-reportado: Poor/Fair/Good/Very Good/Excellent |
| Checkup | Ordinal | Última consulta de rotina |
| Exercise | Nominal (binária) | Prática de exercício físico (Yes/No) |
| Heart_Disease | Nominal (binária) | **TARGET 1** — Doença coronária ou enfarte |
| Skin_Cancer | Nominal (binária) | Diagnóstico de cancro de pele |
| Other_Cancer | Nominal (binária) | Diagnóstico de outro cancro |
| Depression | Nominal (binária) | Diagnóstico de depressão |
| Diabetes | Nominal | **TARGET 2** — Diagnóstico de diabetes |
| Arthritis | Nominal (binária) | Diagnóstico de artrite |
| Sex | Nominal | Sexo (Male/Female) |
| Age_Category | Ordinal | Escalão etário (18-24, 25-29, ..., 80+) |
| Height_(cm) | Rácio | Altura em cm |
| Weight_(kg) | Rácio | Peso em kg |
| BMI | Rácio | Índice de Massa Corporal |
| Smoking_History | Nominal (binária) | Histórico de tabagismo |
| Alcohol_Consumption | Rácio | Freq./quantidade de álcool (autorreportado) |
| Fruit_Consumption | Rácio | Freq./quantidade de fruta |
| Green_Vegetables_Consumption | Rácio | Freq./quantidade de vegetais verdes |
| FriedPotato_Consumption | Rácio | Freq./quantidade de batatas fritas |

## 2.3 Estatísticas Descritivas

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Variáveis numéricas ({len(num_cols)}): {num_cols}')
print(f'Variáveis categóricas ({len(cat_cols)}): {cat_cols}')
print()
df[num_cols].describe().round(2)

## 2.4 Qualidade dos Dados — Valores em Falta, Duplicados e Outliers

In [ ]:
# Distribuição das variáveis categóricas
for col in cat_cols:
    vc  = df[col].value_counts()
    pct = df[col].value_counts(normalize=True) * 100
    print(f'\n── {col} ──')
    print(pd.DataFrame({'Contagem': vc, '%': pct.round(1)}).to_string())

### Distribuição das Variáveis Categóricas Binárias

In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

binary_cols_to_plot = [
    'Heart_Disease', 'Diabetes'
]

n_cols = 4
n_rows = math.ceil(len(binary_cols_to_plot) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(binary_cols_to_plot):
    ax = axes[i]
    counts = df[col].value_counts()
    percentages = df[col].value_counts(normalize=True) * 100

    # Use custom colors for Yes/No, Male/Female, etc. for better distinction
    if col == 'Heart_Disease':
        colors = ['#4CAF50', '#F44336'] # Green for 'No', Red for 'Yes'
        labels_order = ['No', 'Yes']
    elif col == 'Diabetes':
        colors = ['#4CAF50', '#FFC107', '#2196F3', '#F44336'] # Green (No), Amber (Pre-diabetes), Blue (Pregnancy-related), Red (Yes)
        labels_order = ['No', 'No, pre-diabetes or borderline diabetes', 'Yes, but female told only during pregnancy', 'Yes']
        # Filter labels_order to only include categories actually present in the data
        labels_order = [lbl for lbl in labels_order if lbl in counts.index]
        counts = counts.reindex(labels_order)
        percentages = percentages.reindex(labels_order)
    else:
        colors = sns.color_palette('pastel', len(counts))
        labels_order = counts.index.tolist()

    # Reorder counts and percentages based on labels_order if specified
    if labels_order and all(item in counts.index for item in labels_order):
        counts = counts.reindex(labels_order)
        percentages = percentages.reindex(labels_order)
    else:
        # Fallback if specific order is not fully present or for other columns
        labels_order = counts.index.tolist()

    bars = ax.bar(labels_order, counts.values, color=colors, edgecolor='white')
    ax.set_title(f'Distribuição de {col}', fontsize=12)
    ax.set_ylabel('Contagem', fontsize=10)
    ax.tick_params(axis='x', rotation=0)

    for bar, pct in zip(bars, percentages.values):
        height = bar.get_height()
        # Increased vertical offset for the text annotation
        ax.text(bar.get_x() + bar.get_width() / 2, height + (height * 0.25),
                f'{height:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

# Hide any remaining empty subplots
for j in range(len(binary_cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis Categóricas Selecionadas', fontsize=16, y=1.02)
plt.tight_layout()
plt.subplots_adjust(hspace=0.7) # Increased explicit vertical spacing
plt.savefig('fig_selected_binary_bar_charts.png', dpi=120, bbox_inches='tight')
plt.show()

### Distribuição das Variáveis Categóricas Ordinais

In [ ]:
ordinal_cols_to_plot = ['General_Health', 'Checkup']

# Define the natural order for each ordinal variable
ordinal_orders = {
    'General_Health': ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
    'Checkup': ['Never', '5 or more years ago', 'Within the past 2 years', 'Within the past year']
}

n_cols = 2
n_rows = math.ceil(len(ordinal_cols_to_plot) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
axes = axes.flatten()

for i, col in enumerate(ordinal_cols_to_plot):
    ax = axes[i]
    # Ensure the order is respected
    counts = df[col].value_counts().reindex(ordinal_orders[col])
    percentages = (df[col].value_counts(normalize=True) * 100).reindex(ordinal_orders[col])

    bars = ax.bar(counts.index, counts.values, color='skyblue', edgecolor='white')
    ax.set_title(f'Distribuição de {col}', fontsize=12, y=1.1) # Further adjusted y-position of the title
    ax.set_ylabel('Contagem', fontsize=10)
    ax.tick_params(axis='x', rotation=45)

    for bar, pct in zip(bars, percentages.values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + (height * 0.05), # Adjusted offset
                f'{height:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

# Hide any remaining empty subplots
for j in range(len(ordinal_cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis Categóricas Ordinais', fontsize=16, y=1.02)
plt.tight_layout()
plt.subplots_adjust(hspace=0.7)
plt.savefig('fig_ordinal_bar_charts.png', dpi=120, bbox_inches='tight')
plt.show()

### Distribuição da Variável Categórica Ordinal: Categoria de Idade

In [ ]:
age_order = sorted(df['Age_Category'].unique(), key=lambda x: int(x.split('-')[0]) if '-' in x else int(x[:-1]))

plt.figure(figsize=(10, 7)) # Increased figure size slightly for better label fit
ax = sns.countplot(data=df, y='Age_Category', order=age_order, palette='viridis')
plt.title('Distribuição da Categoria de Idade', fontsize=14)
plt.xlabel('Contagem', fontsize=12)
plt.ylabel('Categoria de Idade', fontsize=12)

# Calculate percentages and add annotations
total = len(df)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_width()/total)
    x = p.get_x() + p.get_width() + 0.1 # Adjust x position for text
    y = p.get_y() + p.get_height() / 2 # Center y position
    ax.annotate(percentage, (x, y), ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_age_category_bar_chart.png', dpi=120, bbox_inches='tight')
plt.show()

### Distribuição da Variável Categórica Ordinal: Diabetes (TARGET 2)

In [ ]:
plt.figure(figsize=(10, 6))
db_counts = df['Diabetes'].value_counts().sort_values(ascending=False)
ax = sns.barplot(x=db_counts.index, y=db_counts.values, palette='viridis')

plt.title('Distribuição da Categoria Diabetes (TARGET 2)', fontsize=14)
plt.xlabel('Categoria Diabetes', fontsize=12)
plt.ylabel('Contagem', fontsize=12)
plt.xticks(rotation=15, ha='right')

# Calculate percentages and add annotations
total = len(df)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width() / 2
    y = p.get_height() + (p.get_height() * 0.05) # Adjust y position for text
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig_diabetes_bar_chart.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heart_Disease vs General_Health
gen_health_order = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
pd.crosstab(df['General_Health'], df['Heart_Disease'], normalize='index').loc[gen_health_order].plot(
    kind='bar', stacked=True, ax=axes[0], color=['#2196F3', '#F44336'])
axes[0].set_title('Proporção de Heart Disease por Saúde Geral Auto-Reportada', fontsize=12)
axes[0].set_xlabel('Saúde Geral')
axes[0].set_ylabel('Proporção')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Heart Disease', bbox_to_anchor=(1.05, 1), loc='upper left')

# Heart_Disease vs Diabetes
diabetes_order = ['No', 'No, pre-diabetes or borderline diabetes', 'Yes', 'Yes, but female told only during pregnancy']
pd.crosstab(df['Diabetes'], df['Heart_Disease'], normalize='index').loc[diabetes_order].plot(
    kind='bar', stacked=True, ax=axes[1], color=['#2196F3', '#F44336'])
axes[1].set_title('Proporção de Heart Disease por Categoria de Diabetes', fontsize=12)
axes[1].set_xlabel('Diabetes')
axes[1].set_ylabel('Proporção')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Heart Disease', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.suptitle('Relação entre Variáveis e Heart Disease', y=1.02, fontsize=14)
plt.savefig('fig_var_vs_targets_P1P2.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define CSV_FILE_PATH and load the DataFrame
# This path should match where your CVD_cleaned.csv is located in Google Drive
CSV_FILE_PATH = '/content/drive/MyDrive/CVD_cleaned.csv'
df = pd.read_csv(CSV_FILE_PATH)

# Initialize figure and axes for plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df['Skin_Cancer'].value_counts().plot(kind='bar', ax=axes[0], title='Skin Cancer', color='steelblue')
df['Other_Cancer'].value_counts().plot(kind='bar', ax=axes[1], title='Other Cancer', color='coral')

both = df[(df['Skin_Cancer'] == 'Yes') & (df['Other_Cancer'] == 'Yes')]
print(f"Têm ambos os cancros: {len(both)} ({len(both)/len(df)*100:.1f}%)")
plt.tight_layout(); plt.show()

### Distribuição Visual das Variáveis Categóricas

In [ ]:
# --- Valores em falta ---
missing = df.isna().sum()
print(f'Total de valores em falta: {missing.sum()}')

if missing.sum() == 0:
    print('✓ Não existem valores em falta no dataset.')
else:
    missing_pct = (missing / len(df) * 100).round(2)
    fig, ax = plt.subplots(figsize=(10, 3))
    missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('% de Valores em Falta por Variável')
    ax.set_ylabel('% Missing')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()

# --- Duplicados ---
n_dup = df.duplicated().sum()
print(f'Registos duplicados: {n_dup:,} ({n_dup/len(df)*100:.3f}%)')

In [ ]:
# --- Outliers (método IQR) ---
print('Outliers por variável numérica (método IQR):')
outlier_info = {}
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_info[col] = n_out
    print(f'  {col:<35}: {n_out:>6,} ({n_out/len(df)*100:.2f}%)')

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontsize=8)
    axes[i].set_xticks([])
plt.suptitle('Boxplots das Variáveis Numéricas — Deteção de Outliers', fontsize=12)
plt.tight_layout(); plt.savefig('fig_boxplots.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
print('\nLimites numéricos para Outliers (método IQR):')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f'  {col:<35}: Limite Inferior = {lower_bound:>8.2f}, Limite Superior = {upper_bound:>8.2f}')

In [ ]:
print('\nLimites numéricos para Outliers (método IQR):')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f'  {col:<35}: Limite Inferior = {lower_bound:>8.2f}, Limite Superior = {upper_bound:>8.2f}')

In [ ]:
print('\nLimites numéricos para Outliers (método IQR):')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f'  {col:<35}: Limite Inferior = {lower_bound:>8.2f}, Limite Superior = {upper_bound:>8.2f}')

In [ ]:
# Distribuição dos targets
print("=== Heart_Disease ===")
print(df['Heart_Disease'].value_counts(normalize=True).round(3))

print("\n=== Diabetes ===")
print(df['Diabetes'].value_counts(normalize=True).round(3))

# Valores únicos das categóricas ordinais
for col in ['General_Health', 'Checkup', 'Age_Category', 'Diabetes']:
    print(f"\n=== {col} ===")
    print(df[col].unique())

# Contagem de outliers por variável numérica
limites = {
    'Height_(cm)':                  (140.50, 200.50),
    'Weight_(kg)':                  (27.23,  136.06),
    'BMI':                          (12.75,  43.31),
    'Alcohol_Consumption':          (0,      15.00),
    'Fruit_Consumption':            (0,      57.00),
    'Green_Vegetables_Consumption': (0,      44.00),
    'FriedPotato_Consumption':      (0,      17.00),
}

print("\n=== Contagem de Outliers ===")
for col, (low, high) in limites.items():
    n_low  = (df[col] < low).sum()
    n_high = (df[col] > high).sum()
    total  = n_low + n_high
    pct    = total / len(df) * 100
    print(f"{col:40s}: abaixo={n_low:5d}, acima={n_high:6d}, total={total:6d} ({pct:.2f}%)")

### Interpretação dos Outliers através dos Boxplots e Limites IQR

Para explicar e assumir que os gráficos de *boxplot* traduzem de forma correta os *outliers*, devemos considerar o seguinte:

1.  **Método do Intervalo Interquartil (IQR):** Esta é uma técnica estatística comum para detetar *outliers*. Baseia-se na ideia de que os dados que caem "demasiado longe" da maioria dos dados são considerados anómalos. Especificamente:
    *   **Q1 (Primeiro Quartil):** 25% dos dados estão abaixo deste valor.
    *   **Q3 (Terceiro Quartil):** 75% dos dados estão abaixo deste valor.
    *   **IQR = Q3 - Q1:** Representa a amplitude dos 50% centrais dos dados.

2.  **Cálculo dos Limites:** Um ponto de dado é considerado um *outlier* se estiver:
    *   **Abaixo do Limite Inferior:** Q1 - 1.5 * IQR
    *   **Acima do Limite Superior:** Q3 + 1.5 * IQR
    Os valores que foram calculados no output anterior (`Limite Inferior` e `Limite Superior`) correspondem exatamente a estes pontos de corte para cada variável numérica.

3.  **Representação Visual nos Boxplots:**
    *   As caixas (`box`) nos gráficos representam o intervalo do IQR (de Q1 a Q3), com uma linha dentro da caixa indicando a mediana (Q2).
    *   Os "bigodes" (`whiskers`) estendem-se a partir da caixa até o último ponto de dados que *não* é considerado um *outlier* (ou seja, o valor mais distante dentro dos limites de Q1 - 1.5*IQR e Q3 + 1.5*IQR).
    *   Todos os pontos individuais que são visíveis fora desses "bigodes" nos *boxplots* são os *outliers* identificados pelo método do IQR. São plotados individualmente para chamar a atenção para a sua existência.

Portanto, a combinação dos *boxplots* e dos limites numéricos explicitamente calculados permite-nos afirmar com confiança que os *outliers* estão a ser identificados e representados corretamente de acordo com uma metodologia estatística amplamente aceite.

## 2.5 Distribuição das Variáveis Numéricas

In [ ]:
import math

n_cols = 3
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Média={mean_val:.1f}')
    axes[i].set_title(col); axes[i].set_xlabel('Valor'); axes[i].set_ylabel('Freq.')
    axes[i].legend(fontsize=7)

# esconder subplots extra
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_dist_num.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.6 Distribuição dos Targets e Variáveis Sociodemográficas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heart Disease
hd_counts = df['Heart_Disease'].value_counts()
bars = axes[0].bar(hd_counts.index, hd_counts.values, color=['#2196F3','#F44336'], edgecolor='white')
axes[0].set_title('Distribuição — Heart Disease (Target Principal)', fontweight='bold')
axes[0].set_ylabel('Nº de Registos')
for bar, val in zip(bars, hd_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)

# Diabetes
db_counts = df['Diabetes'].value_counts()
bars2 = axes[1].bar(db_counts.index, db_counts.values,
                    color=sns.color_palette('muted', len(db_counts)), edgecolor='white')
axes[1].set_title('Distribuição — Diabetes (Target Secundário)', fontweight='bold')
axes[1].set_ylabel('Nº de Registos')
plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')
for bar, val in zip(bars2, db_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout(); plt.savefig('fig_targets.png', dpi=120, bbox_inches='tight'); plt.show()

print('⚠ Desequilíbrio de classes (Heart_Disease):')
print((hd_counts / len(df) * 100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição por escalão etário
age_order = sorted(df['Age_Category'].unique())
age_counts = df['Age_Category'].value_counts().reindex(age_order)
axes[0].bar(age_counts.index, age_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição por Escalão Etário')
axes[0].set_ylabel('Nº de Registos')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Distribuição por sexo
sex_counts = df['Sex'].value_counts()
axes[1].pie(sex_counts.values, labels=sex_counts.index,
            autopct='%1.1f%%', colors=['#42A5F5','#EF5350'], startangle=90,
            wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Distribuição por Sexo')

plt.tight_layout(); plt.savefig('fig_age_sex.png', dpi=120, bbox_inches='tight'); plt.show()

## 2.7 Análise de Correlações

### Análise de Correlações entre Variáveis Numéricas

In [ ]:
corr_num = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_num, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de Correlação entre Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.savefig('fig_correlacao_numericas.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Codificação temporária para correlações
df_enc = df.copy()
le = LabelEncoder()

# Mapeamentos ordinais significativos
df_enc['General_Health'] = df_enc['General_Health'].map(
    {'Poor':1,'Fair':2,'Good':3,'Very Good':4,'Excellent':5})
df_enc['Checkup'] = df_enc['Checkup'].map(
    {'Never':0,'5 or more years ago':1,'Within the past 2 to 5 years':2,'Within the past year':3})
age_order_map = {cat:i for i,cat in enumerate(sorted(df['Age_Category'].unique()))}
df_enc['Age_Category'] = df_enc['Age_Category'].map(age_order_map)

binary_cols = ['Exercise','Heart_Disease','Skin_Cancer','Other_Cancer',
               'Depression','Arthritis','Smoking_History']
for c in binary_cols:
    df_enc[c] = df_enc[c].map({'Yes':1,'No':0})
df_enc['Sex'] = df_enc['Sex'].map({'Male':1,'Female':0})
df_enc['Diabetes'] = df_enc['Diabetes'].map(
    {'No':0,'No, pre-diabetes or borderline diabetes':1,'Yes':2}).fillna(
    df_enc['Diabetes'].map(lambda x: le.fit_transform([str(x)])[0] if pd.notna(x) else np.nan))

corr = df_enc.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size':7})
ax.set_title('Matriz de Correlação entre Todas as Variáveis', fontsize=14)
plt.tight_layout(); plt.savefig('fig_correlacao.png', dpi=120, bbox_inches='tight'); plt.show()

print('\n── Correlações com Heart_Disease (ordenadas por valor absoluto) ──')
print(corr['Heart_Disease'].drop('Heart_Disease').sort_values(key=abs, ascending=False).round(3))

---
## 2.8 Análise por Perguntas de Investigação

### P1 — Quais as variáveis mais correlacionadas com Heart Disease?

In [ ]:
# Correlações com Heart_Disease ordenadas
corr_hd = corr['Heart_Disease'].drop('Heart_Disease').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#F44336' if v > 0 else '#2196F3' for v in corr_hd.values]
ax.barh(corr_hd.index, corr_hd.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação de cada variável com Heart_Disease', fontsize=12)
ax.set_xlabel('Correlação de Pearson')
plt.tight_layout(); plt.savefig('fig_corr_hd.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# Prevalência de Heart Disease por escalão etário e sexo
df_plot = df_enc.copy()
df_plot['Age_Category_label'] = df['Age_Category']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

prev_age = df_plot.groupby('Age_Category_label')['Heart_Disease'].mean().reindex(
    sorted(df['Age_Category'].unique())) * 100
prev_age.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Prevalência de Heart Disease por Escalão Etário (%)')
axes[0].set_ylabel('%'); plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

df_plot['Sex_label'] = df['Sex']
prev_sex = df_plot.groupby('Sex_label')['Heart_Disease'].mean() * 100
axes[1].bar(prev_sex.index, prev_sex.values, color=['#42A5F5','#EF5350'], edgecolor='white')
axes[1].set_title('Prevalência de Heart Disease por Sexo (%)')
axes[1].set_ylabel('%')
for bar, val in zip(axes[1].patches, prev_sex.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', va='bottom')

plt.tight_layout(); plt.savefig('fig_prev_age_sex.png', dpi=120, bbox_inches='tight'); plt.show()

> **Conclusão P1:** As variáveis com maior correlação com a doença cardíaca são a idade (`Age_Category`), a artrite (`Arthritis`), o historial de tabagismo (`Smoking_History`), a diabetes e a saúde geral (`General_Health`). A prevalência de DCV aumenta acentuadamente com a idade, confirmando que a população mais idosa é o grupo de maior risco. Os homens apresentam maior prevalência do que as mulheres.

### P2 — Relação entre BMI, Diabetes e Heart Disease

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Boxplot BMI por Heart Disease
hd_yes = df.loc[df['Heart_Disease']=='Yes', 'BMI']
hd_no  = df.loc[df['Heart_Disease']=='No',  'BMI']
axes[0].boxplot([hd_no, hd_yes], labels=['Sem DCV','Com DCV'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Distribuição do BMI — com vs. sem Doença Cardíaca')
axes[0].set_ylabel('BMI')

# Prevalência DCV por categoria de BMI (OMS)
df_bmi = df_enc.copy()
df_bmi['BMI_cat'] = pd.cut(df['BMI'], bins=[0,18.5,25,30,np.inf],
                            labels=['Baixo Peso','Normal','Sobrepeso','Obeso'])
prev_bmi = df_bmi.groupby('BMI_cat', observed=True)['Heart_Disease'].mean() * 100
prev_bmi.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Prevalência de Heart Disease por Categoria BMI (%)')
axes[1].set_ylabel('%'); plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')

plt.tight_layout(); plt.savefig('fig_bmi_hd.png', dpi=120, bbox_inches='tight'); plt.show()

print(f'\nMédia BMI — Com DCV: {hd_yes.mean():.2f} | Sem DCV: {hd_no.mean():.2f}')

In [ ]:
# Scatter plot BMI vs Alcohol — colorido por Heart Disease
sample_scatter = df_enc.sample(5000, random_state=SEED).copy()
sample_scatter['HD_label'] = df.loc[sample_scatter.index, 'Heart_Disease']

fig, ax = plt.subplots(figsize=(8, 5))
for label, color in [('No','steelblue'),('Yes','#F44336')]:
    mask = sample_scatter['HD_label'] == label
    ax.scatter(sample_scatter.loc[mask,'BMI'],
               sample_scatter.loc[mask,'Alcohol_Consumption'],
               alpha=0.3, s=10, color=color, label=label)
ax.set_xlabel('BMI'); ax.set_ylabel('Alcohol Consumption')
ax.set_title('Scatter Plot: BMI vs. Consumo de Álcool (por Heart Disease)')
ax.legend(title='Heart Disease')
plt.tight_layout(); plt.savefig('fig_scatter_bmi_alc.png', dpi=120, bbox_inches='tight'); plt.show()

> **Conclusão P2:** Os indivíduos com doença cardíaca apresentam, em média, um BMI ligeiramente superior. A categoria "Obeso" apresenta a maior prevalência de DCV. No entanto, a relação entre BMI e DCV não é linear e é mediada por outros fatores (como diabetes e idade). O consumo de álcool não apresenta uma relação clara com a DCV no scatter plot, sugerindo que não é um fator discriminativo isolado.

### P3 — Exercício físico e alimentação vs. prevalência de DCV

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Prevalência DCV por exercício
df_enc['Exercise_label'] = df['Exercise']
prev_ex = df_enc.groupby('Exercise_label')['Heart_Disease'].mean() * 100
axes[0].bar(prev_ex.index, prev_ex.values, color=['#F44336','#4CAF50'], edgecolor='white')
axes[0].set_title('Heart Disease (%) por Exercício')
axes[0].set_ylabel('%')
for bar, val in zip(axes[0].patches, prev_ex.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.1,
                 f'{val:.1f}%', ha='center', va='bottom')

# Scatter: Fruit vs Fried Potato
s = df_enc.sample(5000, random_state=SEED)
axes[1].scatter(s['Fruit_Consumption'], s['FriedPotato_Consumption'],
                c=s['Heart_Disease'], cmap='RdBu', alpha=0.3, s=8)
axes[1].set_xlabel('Fruit Consumption')
axes[1].set_ylabel('FriedPotato Consumption')
axes[1].set_title('Fruta vs. Batatas Fritas (cor=Heart Disease)')

# Scatter: Green Veg vs FriedPotato
axes[2].scatter(s['Green_Vegetables_Consumption'], s['FriedPotato_Consumption'],
                c=s['Heart_Disease'], cmap='RdBu', alpha=0.3, s=8)
axes[2].set_xlabel('Green Vegetables Consumption')
axes[2].set_ylabel('FriedPotato Consumption')
axes[2].set_title('Vegetais vs. Batatas Fritas (cor=Heart Disease)')

plt.tight_layout(); plt.savefig('fig_exercicio_alimentacao.png', dpi=120, bbox_inches='tight'); plt.show()

> **Conclusão P3:** A prática de exercício físico está associada a uma prevalência de doença cardíaca mais baixa, o que é consistente com a literatura médica. Quanto à alimentação, os scatter plots não revelam uma separação clara entre grupos com e sem DCV apenas com base no consumo de fruta, vegetais e batatas fritas — o que sugere que estes fatores, isoladamente, têm fraco poder discriminativo e que a DCV é influenciada pela combinação de múltiplos fatores de risco.

### Pairplot das variáveis mais relevantes

In [ ]:
# Pairplot das variáveis numéricas mais correlacionadas com Heart Disease
top_vars = ['BMI', 'Age_Category', 'Alcohol_Consumption', 'Fruit_Consumption', 'FriedPotato_Consumption']
pair_df = df_enc[top_vars + ['Heart_Disease']].sample(3000, random_state=SEED).copy()
pair_df['Heart_Disease'] = df.loc[pair_df.index, 'Heart_Disease']

g = sns.pairplot(pair_df, hue='Heart_Disease', palette={'No':'steelblue','Yes':'#F44336'},
                 plot_kws=dict(alpha=0.3, s=10), diag_kind='kde')
g.fig.suptitle('Pairplot — Variáveis Numéricas por Heart Disease', y=1.02, fontsize=13)
plt.savefig('fig_pairplot.png', dpi=110, bbox_inches='tight'); plt.show()

---
# 3. Preparação dos Dados *(Data Preparation)*

## 3.1 Tratamento de Valores em Falta

In [ ]:
# Lista das variáveis não binárias
vars_nao_binarias = ['Height_(cm)', 'Weight_(kg)', 'BMI',
                      'Alcohol_Consumption', 'Fruit_Consumption',
                      'Green_Vegetables_Consumption', 'FriedPotato_Consumption']

for var in vars_nao_binarias:
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[var] < lower) | (df[var] > upper)]

    print(f"\n{'='*50}")
    print(f"Variável: {var}")
    print(f"Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}")
    print(f"Limite inferior: {lower:.2f} | Limite superior: {upper:.2f}")
    print(f"Nº outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
    print(f"Min real: {df[var].min():.2f} | Max real: {df[var].max():.2f}")
    print(f"Exemplos de outliers:\n{outliers[var].describe()}")

In [ ]:
df_clean = df.copy()

# Variáveis numéricas → imputação pela mediana (robusta a outliers)
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isna().sum() > 0:
        med = df_clean[col].median()
        df_clean[col].fillna(med, inplace=True)
        print(f'  {col}: imputado pela mediana ({med:.2f})')

# Variáveis categóricas → imputação pela moda
for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isna().sum() > 0:
        moda = df_clean[col].mode()[0]
        df_clean[col].fillna(moda, inplace=True)
        print(f'  {col}: imputado pela moda ("{moda}")')

print(f'\nValores em falta após tratamento: {df_clean.isna().sum().sum()}')

## 3.2 Tratamento de Duplicados e transformação de dados

Como identificado na análise exploratória, existem 80 registos redundantes (cópias exatas) que serão removidos para garantir a integridade dos modelos estatísticos.

In [ ]:
n_antes = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
n_depois = len(df_clean)

print(f'Registos antes: {n_antes:,}')
print(f'Registos removidos: {n_antes - n_depois}')
print(f'Registos atuais: {n_depois:,}')

In [ ]:
print('--- Valores Únicos em General_Health ---')
print(df_clean['General_Health'].unique())

print('\n--- Contagem de Frequência ---')
print(df_clean['General_Health'].value_counts())

# Verificar se existem valores nulos inesperados ou strings vazias
print(f"\nValores nulos: {df_clean['General_Health'].isna().sum()}")

In [ ]:
print('--- Valores Únicos e Frequência para Checkup ---')
checkup_counts = df_clean['Checkup'].value_counts()
checkup_pct = df_clean['Checkup'].value_counts(normalize=True) * 100

resumo_checkup = pd.DataFrame({
    'Contagem': checkup_counts,
    'Percentagem (%)': checkup_pct.round(2)
})

display(resumo_checkup)

In [ ]:
print('--- Valores Únicos e Frequência para Diabetes ---')
diabetes_counts = df_clean['Diabetes'].value_counts()
diabetes_pct = df_clean['Diabetes'].value_counts(normalize=True) * 100

resumo_diabetes = pd.DataFrame({
    'Contagem': diabetes_counts,
    'Percentagem (%)': diabetes_pct.round(2)
})

display(resumo_diabetes)

In [ ]:
import pandas as pd

# Garantir que as variáveis de ambiente e dados estão carregados
CSV_FILE_PATH = '/content/drive/MyDrive/CVD_cleaned.csv'

if 'df_clean' not in locals():
    if 'df' not in locals():
        try:
            df = pd.read_csv(CSV_FILE_PATH)
        except:
            print("Erro: Não foi possível carregar o CSV. Verifica se o Google Drive está montado.")
    if 'df' in locals():
        df_clean = df.copy()

# Binarização da variável Diabetes em df_clean
# Classe 1: Apenas diagnósticos confirmados ('Yes')
# Classe 0: Restantes categorias (Ausência, Pré-diabetes, Gestacional)
if 'df_clean' in locals():
    df_clean['Diabetes_Binary'] = df_clean['Diabetes'].apply(lambda x: 1 if x == 'Yes' else 0)

    # Verificação da nova distribuição
    print('--- Distribuição da Variável Diabetes Binarizada ---')
    print(df_clean['Diabetes_Binary'].value_counts())
    print(f'\nPercentagem de Classe Positiva: {(df_clean["Diabetes_Binary"].mean()*100):.2f}%')

    # Mostrar exemplo do mapeamento
    display(df_clean[['Diabetes', 'Diabetes_Binary']].drop_duplicates())
else:
    print("Erro: O dataframe df_clean não pôde ser inicializado.")

In [ ]:
# Filtragem da variável Altura com limites clinicamente plausíveis
n_antes_altura = len(df_clean)

# Aplicar limites: mínimo 140cm e máximo 220cm
df_clean = df_clean[(df_clean['Height_(cm)'] >= 140) & (df_clean['Height_(cm)'] <= 220)]

n_depois_altura = len(df_clean)
removidos = n_antes_altura - n_depois_altura
pct_removidos = (removidos / n_antes_altura) * 100

print(f'--- Tratamento de Outliers: Altura ---')
print(f'Registos removidos: {removidos:,} ({pct_removidos:.2f}%)')
print(f'Registos restantes: {n_depois_altura:,}')

# Verificação dos novos limites
print(f"\nNovo Min: {df_clean['Height_(cm)'].min()} cm")
print(f"Novo Max: {df_clean['Height_(cm)'].max()} cm")

In [ ]:
# Filtragem da variável Peso com limites baseados em critérios clínicos
n_antes_peso = len(df_clean)

# Aplicar limites: mínimo 30kg e máximo 250kg
df_clean = df_clean[(df_clean['Weight_(kg)'] >= 30) & (df_clean['Weight_(kg)'] <= 250)]

n_depois_peso = len(df_clean)
removidos_peso = n_antes_peso - n_depois_peso
pct_removidos_peso = (removidos_peso / n_antes_peso) * 100

print(f'--- Tratamento de Outliers (Critério Clínico): Peso ---')
print(f'Registos removidos: {removidos_peso:,} ({pct_removidos_peso:.4f}%)')
print(f'Registos restantes: {n_depois_peso:,}')

# Verificação dos novos limites e estatísticas rápidas
print(f"\nNovo Min: {df_clean['Weight_(kg)'].min()} kg")
print(f"Novo Max: {df_clean['Weight_(kg)'].max()} kg")
print(f"Mediana atual: {df_clean['Weight_(kg)'].median()} kg")

In [ ]:
import pandas as pd
import numpy as np

# 1. Recalcular o IMC com base nos valores corrigidos de Peso e Altura
# Formula: Peso (kg) / [Altura (m)]^2
df_clean['BMI'] = df_clean['Weight_(kg)'] / ((df_clean['Height_(cm)'] / 100) ** 2)

# 2. Filtragem do IMC com limites clínicos (12 a 60)
n_antes_bmi = len(df_clean)
df_clean = df_clean[(df_clean['BMI'] >= 12) & (df_clean['BMI'] <= 60)]

n_depois_bmi = len(df_clean)
removidos_bmi = n_antes_bmi - n_depois_bmi

# 3. Remover colunas redundantes
df_clean = df_clean.drop(columns=['Height_(cm)', 'Weight_(kg)'])

print('--- Tratamento e Recalculo de IMC ---')
print(f'Registos removidos por IMC implausivel: {removidos_bmi}')
print(f'Registos finais: {len(df_clean):,}')
print('Colunas Weight_(kg) e Height_(cm) removidas.')

# Estatisticas do novo IMC
display(df_clean['BMI'].describe())

In [ ]:
# Tratamento de Outliers para Fruit_Consumption
# O limite clínico de 3 porções por dia (90 por mês) é usado para remover valores implausíveis.
n_antes_fruit = len(df_clean)

df_clean = df_clean[df_clean['Fruit_Consumption'] <= 90]

n_depois_fruit = len(df_clean)
removidos_fruit = n_antes_fruit - n_depois_fruit
pct_removidos_fruit = (removidos_fruit / n_antes_fruit) * 100

print('--- Tratamento de Outliers: Fruit_Consumption ---')
print(f'Registos removidos (> 90): {removidos_fruit:,} ({pct_removidos_fruit:.2f}%)')
print(f'Registos restantes: {len(df_clean):,}')

# Verificação do novo valor máximo
print(f"\nNovo Valor Máximo: {df_clean['Fruit_Consumption'].max()}")
print(f"Mediana atual: {df_clean['Fruit_Consumption'].median()}")

In [ ]:
# Tratamento de Outliers para Green_Vegetables_Consumption
# Seguindo o critério de consistência metodológica com Fruit_Consumption,
# removem-se valores > 90 (mais de 3 porções diárias).
n_antes_veg = len(df_clean)

df_clean = df_clean[df_clean['Green_Vegetables_Consumption'] <= 90]

n_depois_veg = len(df_clean)
removidos_veg = n_antes_veg - n_depois_veg
pct_removidos_veg = (removidos_veg / n_antes_veg) * 100

print('--- Tratamento de Outliers: Green_Vegetables_Consumption ---')
print(f'Registos removidos (> 90): {removidos_veg:,} ({pct_removidos_veg:.2f}%)')
print(f'Registos restantes: {len(df_clean):,}')

# Verificação do novo valor máximo
print(f"\nNovo Valor Máximo: {df_clean['Green_Vegetables_Consumption'].max()}")
print(f"Mediana atual: {df_clean['Green_Vegetables_Consumption'].median()}")

In [ ]:
# Tratamento de Outliers para FriedPotato_Consumption
# O m9étodo IQR identificou valores concentrados em 20 e 30 (erros de escala).
# Optou-se por um limite de 60 (2 porções/dia) para manter hábitos plausíveis.
n_antes_potato = len(df_clean)

df_clean = df_clean[df_clean['FriedPotato_Consumption'] <= 60]

n_depois_potato = len(df_clean)
removidos_potato = n_antes_potato - n_depois_potato
pct_removidos_potato = (removidos_potato / n_antes_potato) * 100

print('--- Tratamento de Outliers: FriedPotato_Consumption ---')
print(f'Registos removidos (> 60): {removidos_potato:,} ({pct_removidos_potato:.2f}%)')
print(f'Registos restantes: {len(df_clean):,}')

# Verificação do novo valor máximo
print(f"\nNovo Valor Máximo: {df_clean['FriedPotato_Consumption'].max()}")
print(f"Mediana atual: {df_clean['FriedPotato_Consumption'].median()}")

## 3.3 Dataset Completamente Categórico

As variáveis numéricas contínuas são discretizadas com critérios clínicos (BMI) ou por *equal-frequency* (restantes), de forma a preservar a distribuição dos dados.

In [ ]:
import pandas as pd
import numpy as np

dataset_categorico = df_clean.copy()

# 1. Categorização do IMC (BMI)
dataset_categorico['BMI'] = pd.cut(
    dataset_categorico['BMI'],
    bins=[0, 18.5, 24.9, 29.9, np.inf],
    labels=['Underweight', 'Normal weight', 'Overweight', 'Obese'],
    right=True
)

# 2. Categorização das variáveis de consumo com ajuste dinâmico de labels
vars_consumo = ['Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption']

for var in vars_consumo:
    bins = pd.qcut(dataset_categorico[var], q=3, duplicates='drop', retbins=True)[1]
    n_bins = len(bins) - 1

    if n_bins == 3:
        current_labels = ['Low', 'Medium', 'High']
    elif n_bins == 2:
        current_labels = ['Low', 'High']
    else:
        current_labels = None

    dataset_categorico[var] = pd.qcut(
        dataset_categorico[var],
        q=3,
        labels=current_labels,
        duplicates='drop'
    )

# 3. Transformar a coluna Diabetes em binária (Yes/No) usando a lógica do Diabetes_Binary
# No df_clean, Diabetes_Binary já foi definido como 1 se 'Yes' e 0 para o resto.
dataset_categorico['Diabetes'] = dataset_categorico['Diabetes_Binary'].map({1: 'Yes', 0: 'No'})

# 4. Remover colunas auxiliares para garantir as 17 variáveis originais
exclude_cols = ['Weight_(kg)', 'Height_(cm)', 'Diabetes_Binary']
filt_cols = [c for c in dataset_categorico.columns if c not in exclude_cols]
dataset_categorico = dataset_categorico[filt_cols]

print(f'Dataset categórico "dataset_categorico" criado com sucesso.')
print(f'Número de variáveis: {len(dataset_categorico.columns)}')
print(f'Valores na coluna Diabetes: {dataset_categorico["Diabetes"].unique()}')
display(dataset_categorico.head())

In [ ]:
# Distribuição das variáveis categorizadas com percentagens
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
total_rows = len(dataset_categorico)

def add_percentage_labels(ax, total):
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 5), textcoords='offset points')

# BMI
dataset_categorico['BMI'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title('BMI (Categorizado)'); plt.setp(axes[0,0].get_xticklabels(), rotation=20, ha='right')
add_percentage_labels(axes[0,0], total_rows)

# Alcohol Consumption
dataset_categorico['Alcohol_Consumption'].value_counts().plot(kind='bar', ax=axes[0,1], color='darkorange', edgecolor='white')
axes[0,1].set_title('Alcohol Consumption')
add_percentage_labels(axes[0,1], total_rows)

# Fruit Consumption
dataset_categorico['Fruit_Consumption'].value_counts().plot(kind='bar', ax=axes[0,2], color='seagreen', edgecolor='white')
axes[0,2].set_title('Fruit Consumption')
add_percentage_labels(axes[0,2], total_rows)

# Green Vegetables Consumption
dataset_categorico['Green_Vegetables_Consumption'].value_counts().plot(kind='bar', ax=axes[1,0], color='forestgreen', edgecolor='white')
axes[1,0].set_title('Green Vegetables Consumption')
add_percentage_labels(axes[1,0], total_rows)

# Fried Potato Consumption
dataset_categorico['FriedPotato_Consumption'].value_counts().plot(kind='bar', ax=axes[1,1], color='goldenrod', edgecolor='white')
axes[1,1].set_title('Fried Potato Consumption')
add_percentage_labels(axes[1,1], total_rows)

# Diabetes (Binarizado)
dataset_categorico['Diabetes'].value_counts().plot(kind='bar', ax=axes[1,2], color=['#2196F3', '#F44336'], edgecolor='white')
axes[1,2].set_title('Diabetes (Target 2 - Binarizado)')
add_percentage_labels(axes[1,2], total_rows)

plt.tight_layout(); plt.savefig('fig_cat_dist_expanded_pct.png', dpi=120, bbox_inches='tight'); plt.show()

##3.4 Dataset Completamente Numérico

As variáveis categóricas são convertidas para valores numéricos, respeitando a **ordem** onde existe (ordinais) e usando **Label Encoding** para as nominais binárias.

In [ ]:
import pandas as pd
import numpy as np

# Criar cópia a partir do dataset_categorico
df_num = dataset_categorico.copy()

# 1. Mapeamento de Variáveis Ordinais (Mantendo a Ordem)
# General Health
df_num['General_Health'] = df_num['General_Health'].map(
    {'Poor': 1, 'Fair': 2, 'Good': 3, 'Very Good': 4, 'Excellent': 5})

# Checkup
df_num['Checkup'] = df_num['Checkup'].map(
    {'Never': 0, '5 or more years ago': 1, 'Within the past 5 years': 2, 'Within the past 2 years': 3, 'Within the past year': 4})

# Age Category (Ordem alfabética coincide com a cronológica neste dataset)
age_order = sorted(dataset_categorico['Age_Category'].unique())
df_num['Age_Category'] = df_num['Age_Category'].map({cat: i for i, cat in enumerate(age_order)})

# BMI (Categorizado no passo anterior)
df_num['BMI'] = df_num['BMI'].map(
    {'Underweight': 1, 'Normal weight': 2, 'Overweight': 3, 'Obese': 4})

# Variáveis de Consumo (Low, Medium, High)
labels_consumo = {'Low': 1, 'Medium': 2, 'High': 3}
for col in ['Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption']:
    df_num[col] = df_num[col].map(labels_consumo)

# 2. Mapeamento de Variáveis Nominais / Binárias
# Sex (One-Hot simplificado para binária)
df_num['Sex'] = df_num['Sex'].map({'Male': 1, 'Female': 0})

# Health Flags (Yes/No -> 1/0)
binary_cols = ['Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Smoking_History']
for col in binary_cols:
    df_num[col] = df_num[col].map({'Yes': 1, 'No': 0})

print('Dataset numérico (df_num) gerado a partir do dataset_categorico.')
print(f'Dimensões: {df_num.shape}')
display(df_num.head())

## 3.5 Normalização

São aplicados dois métodos de normalização ao dataset numérico:

- **Min-Max** → valores em [0,1]; preserva a forma da distribuição; sensível a outliers
- **Z-score** → média 0, desvio padrão 1; mais robusta a outliers; utilizada por defeito no clustering

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Definir as features (todas exceto o target principal)
features = [c for c in df_num.columns if c != 'Heart_Disease']
X = df_num[features].copy()

# 2. Inicializar e aplicar o MinMaxScaler
scaler_mm = MinMaxScaler()
X_minmax = pd.DataFrame(scaler_mm.fit_transform(X), columns=features)

# 3. Adicionar o target de volta para exploração futura, se necessário
df_num_scaled = X_minmax.copy()
df_num_scaled['Heart_Disease'] = df_num['Heart_Disease'].values

print('Dataset Numérico Escalado (Min-Max) criado com sucesso.')
print(f'Intervalo dos dados: [{df_num_scaled[features].min().min()}, {df_num_scaled[features].max().max()}]')
display(df_num_scaled.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Definir as features (todas exceto o target principal)
features = [c for c in df_num.columns if c != 'Heart_Disease']
X = df_num[features].copy()

# 2. Inicializar e aplicar o StandardScaler (Z-score)
scaler_zs = StandardScaler()
X_zscore = pd.DataFrame(scaler_zs.fit_transform(X), columns=features)

# 3. Adicionar o target de volta
df_num_zscore = X_zscore.copy()
df_num_zscore['Heart_Disease'] = df_num['Heart_Disease'].values

print('Dataset Num&eacute;rico Normalizado (StandardScaler) criado com sucesso.')
print(f'M&eacute;dia aproximada: {df_num_zscore[features].mean().mean():.2f}')
print(f'Desvio padr&atilde;o aproximado: {df_num_zscore[features].std().mean():.2f}')
display(df_num_zscore.head())

## 3.6 Amostragem e Projeção PCA para Visualização

In [ ]:
N_SAMPLE = 10000
sample_idx = np.random.choice(len(X_zscore), size=N_SAMPLE, replace=False)
X_sample     = X_zscore.iloc[sample_idx].reset_index(drop=True)
X_mm_sample  = X_minmax.iloc[sample_idx].reset_index(drop=True)

pca2 = PCA(n_components=2, random_state=SEED)
X_pca_full   = pca2.fit_transform(X_zscore)
X_pca_sample = X_pca_full[sample_idx]

var_exp = pca2.explained_variance_ratio_
print(f'Variância explicada — PC1: {var_exp[0]*100:.1f}%  |  PC2: {var_exp[1]*100:.1f}%  |  Total: {var_exp.sum()*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_pca_sample[:,0], X_pca_sample[:,1], alpha=0.2, s=6, color='steelblue')
ax.set_title('Projeção PCA 2D — Dataset Normalizado (amostra 10k)')
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
plt.tight_layout(); plt.savefig('fig_pca_base.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:


loadings = pd.DataFrame(
    pca2.components_.T,
    index=X_zscore.columns,
    columns=['PC1', 'PC2']
).sort_values('PC1', key=abs, ascending=False)

print(loadings)

---
# 4. Modelação — Abordagens Não Supervisionadas

## 4.1 K-Means

### 4.1.1 Escolha do k — Método do Cotovelo e Silhouette

## K-Means Clustering Analysis

Para a análise de clustering com K-Means, a variável `Heart_Disease` foi excluída do conjunto de dados,
de forma a evitar que esta influenciasse a identificação de perfis naturais nos dados.

Foram testados valores de k entre 2 e 10, avaliando o desempenho através de duas métricas:

- **Silhouette Score:** atingiu o valor mais alto em k=3 (0.1173), sugerindo que 3 clusters
  é a escolha mais adequada para este conjunto de dados.
- **Inércia (WCSS):** apresenta uma diminuição progressiva, com um "cotovelo" ligeiramente
  visível em k=3 ou k=4, o que corrobora a escolha de k=3.

Com base nestes resultados, o modelo final de K-Means será aplicado com **k=3**.

In [ ]:
K_range   = range(2, 11)
inertias  = []
silhs     = []

# Filtrar a variável Heart_Disease para não influenciar os clusters
features_clustering = [c for c in X_sample.columns if c != 'Heart_Disease']
X_train = X_sample[features_clustering]

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbs = km.fit_predict(X_train)
    inertias.append(km.inertia_)
    silhs.append(silhouette_score(X_train, lbs, sample_size=3000, random_state=SEED))
    print(f'  k={k:2d}  inertia={km.inertia_:>10.0f}  silhouette={silhs[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Método do Cotovelo — Inércia (WCSS) vs. k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inércia')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

axes[1].plot(list(K_range), silhs, 'rs-', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score vs. k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('K-Means — Escolha do k ótimo (Sem Heart_Disease, amostra 10k)', fontsize=13)
plt.tight_layout(); plt.savefig('fig_elbow.png', dpi=120, bbox_inches='tight'); plt.show()

### 4.1.2 K-Means com k ótimo

## Conclusões do K-Means (k=3)

O modelo final de K-Means foi aplicado com k=3, normalização Z-score e o dataset completo
(excluindo `Heart_Disease`). As métricas obtidas sugerem uma separação moderada entre clusters:

- **Silhouette Score de 0.1162** — indica alguma estrutura nos dados, mas com sobreposição
  entre clusters, o que é comum em dados de saúde com variáveis muito correlacionadas.
- **Davies-Bouldin Index de 2.6443** — valor relativamente alto, sugerindo que os clusters
  não são muito compactos nem bem separados entre si.
- **Calinski-Harabasz Index de 28257.2** — valor elevado, indicando boa densidade interna
  dos clusters relativamente à separação entre eles.

### Perfis identificados

Os três clusters apresentam tamanhos bastante desequilibrados (9.7%, 36.1% e 54.1%),
o que sugere que a população não se divide em grupos uniformes. Com base nos valores
médios das variáveis, é possível caracterizar cada cluster:

- **Cluster 0 (9.7%)** — Perfil de indivíduos com elevada prevalência de cancro de pele
  (100%), sendo um grupo muito específico e minoritário.
- **Cluster 1 (36.1%)** — Perfil de maior risco, com valores mais elevados de Diabetes
  e Artrite, sugerindo indivíduos com múltiplas comorbilidades.
- **Cluster 2 (54.1%)** — Perfil mais saudável, com melhor saúde geral, maior prática
  de exercício e menor prevalência de Diabetes.

Estes perfis serão posteriormente cruzados com a variável `Heart_Disease` para analisar
a sua distribuição em cada cluster.

In [ ]:
K_BEST = int(np.argmax(silhs)) + 2
print(f'k escolhido (maior Silhouette na amostra): {K_BEST}')

km_final = KMeans(n_clusters=K_BEST, random_state=SEED, n_init=10)
labels_km = km_final.fit_predict(X_zscore)

sil_km  = silhouette_score(X_zscore, labels_km, sample_size=5000, random_state=SEED)
dbi_km  = davies_bouldin_score(X_zscore, labels_km)
chi_km  = calinski_harabasz_score(X_zscore, labels_km)

print(f'\nMétricas K-Means (k={K_BEST}, Z-score, dataset completo):')
print(f'  Silhouette Score:        {sil_km:.4f}  (↑ melhor)')
print(f'  Davies-Bouldin Index:    {dbi_km:.4f}  (↓ melhor)')
print(f'  Calinski-Harabasz Index: {chi_km:.1f}  (↑ melhor)')

uniq, cnts = np.unique(labels_km, return_counts=True)
print('\nTamanho dos clusters:')
for u, c in zip(uniq, cnts):
    print(f'  Cluster {u}: {c:>7,} ({c/len(df_num)*100:.1f}%)')

df_num['Cluster_KM'] = labels_km
cluster_profiles = df_num.drop(columns=['Heart_Disease'], errors='ignore').groupby('Cluster_KM').mean(numeric_only=True)
print(cluster_profiles.T.round(2))

Apesar de a variável Skin_Cancer apresentar correlação positiva com a doença cardíaca, verificou-se que esta dominava o processo de clustering, levando à formação de um cluster quase exclusivamente definido pela sua presença.

Dado que o objetivo da análise é identificar perfis de risco cardiovascular mais abrangentes, esta variável foi removida numa segunda iteração, permitindo ao algoritmo capturar padrões mais complexos relacionados com idade, diabetes e estilo de vida.

## Iteração 2: Remoção da variável Skin_Cancer

Analisando os perfis da primeira iteração, verificou-se que o Cluster 0 era quase
inteiramente definido pela presença de cancro de pele (Skin_Cancer = 1.00), enquanto
os restantes clusters apresentavam Skin_Cancer = 0.00.

Isto indica que o algoritmo utilizou esta variável como critério dominante de separação,
em detrimento de padrões mais relevantes para o risco cardiovascular.

Dado que o objetivo é identificar perfis de saúde mais abrangentes, a variável
`Skin_Cancer` foi removida e o clustering foi repetido, permitindo ao algoritmo
capturar padrões mais ricos relacionados com idade, diabetes, estilo de vida
e outras comorbilidades.

Embora k=2 apresente o Silhouette Score mais elevado (0.1116), a diferença para k=3 (0.1107) é marginal. Optou-se por k=3 por permitir uma segmentação mais rica e interpretável dos perfis de saúde, alinhada com os objetivos do projeto.

In [ ]:
# Avaliar k ótimo para iteração sem Skin_Cancer
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt

k_range = range(2, 11)
inertias_iter = []
silhouettes_iter = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_zscore_sem_skin)

    inertias_iter.append(km.inertia_)
    silhouettes_iter.append(
        silhouette_score(X_zscore_sem_skin, labels, sample_size=5000, random_state=SEED)
    )
    print(f'k={k} | Silhouette: {silhouettes_iter[-1]:.4f} | Inércia: {inertias_iter[-1]:.2f}')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias_iter, marker='o', color='steelblue')
axes[0].set_title('Método do Cotovelo (sem Skin_Cancer)')
axes[0].set_xlabel('Número de Clusters (k)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].grid(True)

axes[1].plot(k_range, silhouettes_iter, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score (sem Skin_Cancer)')
axes[1].set_xlabel('Número de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Melhor k
best_k_iter = k_range[silhouettes_iter.index(max(silhouettes_iter))]
print(f'\nMelhor k por Silhouette Score: {best_k_iter}')

In [ ]:
# ============================================================
# K-MEANS FINAL — SEM Heart_Disease E SEM Skin_Cancer
# Heart_Disease é usado apenas depois para avaliar o risco
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

SEED = 42

# ------------------------------------------------------------
# 1) Definir variáveis a excluir do clustering
# ------------------------------------------------------------

excluir_clustering = ['Heart_Disease', 'Skin_Cancer']

features_clustering = [
    col for col in df_num.columns
    if col not in excluir_clustering
    and not col.startswith('Cluster')
]

print("Variáveis usadas no clustering:")
print(features_clustering)
print("\nNúmero de variáveis usadas:", len(features_clustering))


# ------------------------------------------------------------
# 2) Normalização Z-score
# ------------------------------------------------------------

scaler = StandardScaler()

X_cluster = pd.DataFrame(
    scaler.fit_transform(df_num[features_clustering]),
    columns=features_clustering,
    index=df_num.index
)


# ------------------------------------------------------------
# 3) Escolha de k — testar vários valores
# ------------------------------------------------------------

resultados_k = []

for k in range(2, 8):
    km_temp = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=10
    )

    labels_temp = km_temp.fit_predict(X_cluster)

    resultados_k.append({
        'k': k,
        'Inercia': km_temp.inertia_,
        'Silhouette': silhouette_score(X_cluster, labels_temp),
        'Davies_Bouldin': davies_bouldin_score(X_cluster, labels_temp),
        'Calinski_Harabasz': calinski_harabasz_score(X_cluster, labels_temp)
    })

resultados_k = pd.DataFrame(resultados_k)

display(resultados_k.round(4))


# ------------------------------------------------------------
# 4) Gráfico cotovelo e silhouette
# ------------------------------------------------------------

plt.figure(figsize=(7, 4))
plt.plot(resultados_k['k'], resultados_k['Inercia'], marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo — K-Means sem Heart_Disease e Skin_Cancer')
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(resultados_k['k'], resultados_k['Silhouette'], marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette por k — K-Means sem Heart_Disease e Skin_Cancer')
plt.grid(True)
plt.show()




In [ ]:
# ------------------------------------------------------------
# 5) Modelo final com k = 3
# ------------------------------------------------------------

k_final = 3

km_final = KMeans(
    n_clusters=k_final,
    random_state=SEED,
    n_init=10
)

df_num['Cluster_Final'] = km_final.fit_predict(X_cluster)

labels_final = df_num['Cluster_Final']


# ------------------------------------------------------------
# 6) Métricas finais
# ------------------------------------------------------------

sil = silhouette_score(X_cluster, labels_final)
db = davies_bouldin_score(X_cluster, labels_final)
ch = calinski_harabasz_score(X_cluster, labels_final)

print("Métricas finais — K-Means sem Heart_Disease e sem Skin_Cancer")
print("Silhouette:", round(sil, 4))
print("Davies-Bouldin:", round(db, 4))
print("Calinski-Harabasz:", round(ch, 2))


# ------------------------------------------------------------
# 7) Caracterização dos clusters
# Heart_Disease entra só aqui, para avaliar risco
# ------------------------------------------------------------

perfil_clusters = df_num.groupby('Cluster_Final')[
    features_clustering + ['Heart_Disease']
].mean().round(2)

display(perfil_clusters)


# ------------------------------------------------------------
# 8) Tamanho dos clusters e risco cardiovascular
# ------------------------------------------------------------

resumo_risco = df_num.groupby('Cluster_Final').agg(
    n=('Heart_Disease', 'size'),
    percentagem=('Heart_Disease', lambda x: round(len(x) / len(df_num) * 100, 1)),
    risco_heart_disease=('Heart_Disease', 'mean'),
    diabetes=('Diabetes', 'mean'),
    idade_media=('Age_Category', 'mean'),
    saude_geral=('General_Health', 'mean'),
    exercicio=('Exercise', 'mean'),
    bmi=('BMI', 'mean')
).round(3)

display(resumo_risco)


# ------------------------------------------------------------
# 9) Nomear clusters automaticamente pelo risco de Heart_Disease
# ------------------------------------------------------------

ordem_risco = (
    df_num.groupby('Cluster_Final')['Heart_Disease']
    .mean()
    .sort_values()
    .index
)

nomes_clusters = {
    ordem_risco[0]: 'Baixo risco - jovens saudáveis',
    ordem_risco[1]: 'Risco moderado - mais velhos',
    ordem_risco[2]: 'Alto risco - diabéticos/comorbilidades'
}

df_num['Cluster_Nome'] = df_num['Cluster_Final'].map(nomes_clusters)

print("Mapeamento dos clusters:")
print(nomes_clusters)


# ------------------------------------------------------------
# 10) Resumo final com nomes dos perfis
# ------------------------------------------------------------

resumo_nomeado = df_num.groupby('Cluster_Nome').agg(
    n=('Heart_Disease', 'size'),
    percentagem=('Heart_Disease', lambda x: round(len(x) / len(df_num) * 100, 1)),
    risco_heart_disease=('Heart_Disease', 'mean'),
    diabetes=('Diabetes', 'mean'),
    idade_media=('Age_Category', 'mean'),
    saude_geral=('General_Health', 'mean'),
    exercicio=('Exercise', 'mean'),
    bmi=('BMI', 'mean'),
    smoking=('Smoking_History', 'mean'),
    depression=('Depression', 'mean'),
    arthritis=('Arthritis', 'mean')
).round(3)

display(resumo_nomeado)


# ------------------------------------------------------------
# 11) PCA para visualização dos clusters
# ------------------------------------------------------------

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_cluster)

df_pca = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2'],
    index=df_num.index
)

df_pca['Cluster_Nome'] = df_num['Cluster_Nome']

print("Variância explicada pelas duas primeiras componentes:")
print(round(pca.explained_variance_ratio_[0] * 100, 2), "%")
print(round(pca.explained_variance_ratio_[1] * 100, 2), "%")
print("Total:", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

plt.figure(figsize=(8, 6))

for nome in df_pca['Cluster_Nome'].unique():
    subset = df_pca[df_pca['Cluster_Nome'] == nome]
    plt.scatter(
        subset['PC1'],
        subset['PC2'],
        label=nome,
        alpha=0.4,
        s=10
    )

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Visualização PCA — Clusters finais')
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 12) Comparar risco por cluster em percentagem
# ------------------------------------------------------------

risco_percentagem = (
    df_num.groupby('Cluster_Nome')['Heart_Disease']
    .mean()
    .sort_values()
    * 100
).round(2)

display(risco_percentagem)

plt.figure(figsize=(8, 4))
risco_percentagem.plot(kind='bar')
plt.ylabel('% com Heart Disease')
plt.xlabel('Perfil')
plt.title('Prevalência de Heart Disease por perfil de cluster')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y')
plt.show()

In [ ]:
# K-Means sem Skin_Cancer — iteração CRISP-DM
features_sem_skin = [c for c in features if c != 'Skin_Cancer']

# Garantir que todas as variáveis usadas são numéricas
cols_perfil = features_sem_skin # Removido 'Heart_Disease' da análise de perfilagem

for col in cols_perfil:
    if df_num[col].dtype.name == "category":
        df_num[col] = df_num[col].cat.codes

# Caso Heart_Disease seja texto (Yes/No), converter explicitamente
# Esta linha é redundante, pois Heart_Disease já foi convertida para numérica (0/1) anteriormente
# e foi removida de 'cols_perfil' para esta análise.
# if df_num['Heart_Disease'].dtype == 'object':
#     df_num['Heart_Disease'] = df_num['Heart_Disease'].map({'No': 0, 'Yes': 1})

# Normalização
scaler_iter = StandardScaler()
X_zscore_sem_skin = pd.DataFrame(
    scaler_iter.fit_transform(df_num[features_sem_skin]),
    columns=features_sem_skin
)

# K-Means
km_sem_skin = KMeans(n_clusters=3, random_state=SEED, n_init=10)
labels_sem_skin = km_sem_skin.fit_predict(X_zscore_sem_skin)

# Perfis dos clusters
df_num['Cluster_iter'] = labels_sem_skin

perfil_iter = (
    df_num
    .groupby('Cluster_iter')[cols_perfil] # Agora 'cols_perfil' não inclui 'Heart_Disease'
    .mean(numeric_only=True)
    .round(2)
)

print(perfil_iter.T)

# Métricas
sil_iter = silhouette_score(
    X_zscore_sem_skin,
    labels_sem_skin,
    sample_size=5000,
    random_state=SEED
)

dbi_iter = davies_bouldin_score(X_zscore_sem_skin, labels_sem_skin)

print(f'\nSilhouette: {sil_iter:.4f} | Davies-Bouldin: {dbi_iter:.4f}')

In [ ]:
print("--- K-Means com dados SEM normalização ---")

# 1. Preparar dados não normalizados (amostra para Elbow/Silhouette)
features_unnorm = [c for c in df_num.columns if c != 'Heart_Disease']
X_unnorm = df_num[features_unnorm].copy()

# Usar a mesma amostra para consistência
unnorm_sample_idx = np.random.choice(len(X_unnorm), size=N_SAMPLE, replace=False)
X_unnorm_sample = X_unnorm.iloc[unnorm_sample_idx].reset_index(drop=True)

# 2. Escolha do k — Método do Cotovelo e Silhouette para dados não normalizados
K_range_unnorm   = range(2, 11)
inertias_unnorm  = []
silhs_unnorm     = []

for k in K_range_unnorm:
    km_unnorm_sampler = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbs_unnorm_sampler = km_unnorm_sampler.fit_predict(X_unnorm_sample)
    inertias_unnorm.append(km_unnorm_sampler.inertia_)
    silhs_unnorm.append(silhouette_score(X_unnorm_sample, lbs_unnorm_sampler, sample_size=3000, random_state=SEED))
    print(f'  k={k:2d} (unnorm) inertia={km_unnorm_sampler.inertia_:>10.0f}  silhouette={silhs_unnorm[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_range_unnorm), inertias_unnorm, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Método do Cotovelo — Inércia (WCSS) vs. k (Sem Normalização)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inércia')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

axes[1].plot(list(K_range_unnorm), silhs_unnorm, 'rs-', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score vs. k (Sem Normalização)')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('K-Means — Escolha do k ótimo (Sem Normalização, amostra 10k)', fontsize=13)
plt.tight_layout(); plt.show()

# 3. K-Means com k=3 para dados não normalizados (dataset completo)
print("\n--- K-Means com k=3 para dados SEM normalização (dataset completo) ---")
km_unnorm = KMeans(n_clusters=3, random_state=SEED, n_init=10)
labels_km_unnorm = km_unnorm.fit_predict(X_unnorm)

sil_unnorm = silhouette_score(X_unnorm, labels_km_unnorm, sample_size=5000, random_state=SEED)
dbi_unnorm = davies_bouldin_score(X_unnorm, labels_km_unnorm)
chi_unnorm = calinski_harabasz_score(X_unnorm, labels_km_unnorm)

print(f'\nMétricas K-Means (k=3, SEM normalização, dataset completo):')
print(f'  Silhouette Score:        {sil_unnorm:.4f}  (↑ melhor)')
print(f'  Davies-Bouldin Index:    {dbi_unnorm:.4f}  (↓ melhor)')
print(f'  Calinski-Harabasz Index: {chi_unnorm:.1f}  (↑ melhor)')

uniq_unnorm, cnts_unnorm = np.unique(labels_km_unnorm, return_counts=True)
print('\nTamanho dos clusters (SEM normalização):')
for u, c in zip(uniq_unnorm, cnts_unnorm):
    print(f'  Cluster {u}: {c:>7,} ({c/len(X_unnorm)*100:.1f}%)')

df_num_unnorm_clustered = X_unnorm.copy()
df_num_unnorm_clustered['Cluster_KM_Unnorm'] = labels_km_unnorm
cluster_profiles_unnorm = df_num_unnorm_clustered.groupby('Cluster_KM_Unnorm').mean(numeric_only=True)
print('\nPerfil dos clusters (SEM normalização):')
print(cluster_profiles_unnorm.T.round(2))

# 4. Comparar os perfis dos clusters e as métricas com os do Z-score (já feito no código anterior)
# A comparação direta de métricas para Min-Max e Z-score já foi realizada no passo anterior (cell AjD4nEmO1dkS).

# 5. Visualizar clusters com PCA (Z-score)
print("\n--- Visualização de clusters K-Means (k=3) com Projeção PCA (Z-score) ---")
# User provided code
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(X_pca_sample[:,0], X_pca_sample[:,1],
                     c=labels_km[sample_idx], cmap='tab10',
                     alpha=0.3, s=6)
ax.legend(*scatter.legend_elements(), title='Cluster')
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
ax.set_title('K-Means (k=3) — Projeção PCA 2D (Z-score)')
plt.tight_layout(); plt.show()

# Adicionalmente, visualizar os clusters K-Means para dados não normalizados com PCA
# Realizar PCA na amostra de dados não normalizados
pca_unnorm = PCA(n_components=2, random_state=SEED)
X_pca_unnorm_sample = pca_unnorm.fit_transform(X_unnorm.iloc[unnorm_sample_idx])
var_exp_unnorm = pca_unnorm.explained_variance_ratio_

print("\n--- Visualização de clusters K-Means (k=3) com Projeção PCA (SEM normalização) ---")
fig, ax = plt.subplots(figsize=(8, 5))
scatter_unnorm = ax.scatter(X_pca_unnorm_sample[:,0], X_pca_unnorm_sample[:,1],
                     c=labels_km_unnorm[unnorm_sample_idx], cmap='tab10',
                     alpha=0.3, s=6)
ax.legend(*scatter_unnorm.legend_elements(), title='Cluster')
ax.set_xlabel(f'PC1 ({var_exp_unnorm[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp_unnorm[1]*100:.1f}%)')
ax.set_title('K-Means (k=3) — Projeção PCA 2D (SEM normalização)')
plt.tight_layout(); plt.show()

In [ ]:
# Visualização PCA 2D
lbs_sample = labels_km[sample_idx]
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(X_pca_sample[:,0], X_pca_sample[:,1],
                c=lbs_sample, cmap='tab10', alpha=0.4, s=8)
ax.set_title(f'K-Means (k={K_BEST}) — Projeção PCA 2D (amostra 10k)')
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.savefig('fig_kmeans_pca.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
print('--- K-Means com dados normalizados MinMaxScaler ---')

# 1. Amostra de 10k (X_mm_sample) para testar k de 2 a 10
K_range_mm   = range(2, 11)
inertias_mm  = []
silhs_mm     = []

for k in K_range_mm:
    km_mm_sampler = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbs_mm_sampler = km_mm_sampler.fit_predict(X_mm_sample) # Use X_mm_sample
    inertias_mm.append(km_mm_sampler.inertia_)
    silhs_mm.append(silhouette_score(X_mm_sample, lbs_mm_sampler, sample_size=3000, random_state=SEED))
    print(f'  k={k:2d} (MinMax) inertia={km_mm_sampler.inertia_:>10.0f}  silhouette={silhs_mm[-1]:.4f}')

# 2. Plotar elbow (inércia) e Silhouette Score lado a lado
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_range_mm), inertias_mm, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Método do Cotovelo — Inércia (WCSS) vs. k (MinMaxScaler)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inércia')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

axes[1].plot(list(K_range_mm), silhs_mm, 'rs-', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score vs. k (MinMaxScaler)')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('K-Means — Escolha do k ótimo (MinMaxScaler, amostra 10k)', fontsize=13)
plt.tight_layout(); plt.savefig('fig_elbow_minmax.png', dpi=120, bbox_inches='tight'); plt.show()

# 3. Escolher o k com maior Silhouette
K_BEST_MM = int(np.argmax(silhs_mm)) + 2
print(f'k escolhido (maior Silhouette na amostra MinMaxScaler): {K_BEST_MM}')

# 4. Treinar K-Means com esse k no dataset completo (X_minmax)
km_final_mm = KMeans(n_clusters=K_BEST_MM, random_state=SEED, n_init=10)
labels_km_mm_full = km_final_mm.fit_predict(X_minmax)

# 5. Calcular métricas: Silhouette, Davies-Bouldin, Calinski-Harabasz
sil_km_mm  = silhouette_score(X_minmax, labels_km_mm_full, sample_size=5000, random_state=SEED)
dbi_km_mm  = davies_bouldin_score(X_minmax, labels_km_mm_full)
chi_km_mm  = calinski_harabasz_score(X_minmax, labels_km_mm_full)

print(f'\nMétricas K-Means (k={K_BEST_MM}, MinMaxScaler, dataset completo):')
print(f'  Silhouette Score:        {sil_km_mm:.4f}  (↑ melhor)')
print(f'  Davies-Bouldin Index:    {dbi_km_mm:.4f}  (↓ melhor)')
print(f'  Calinski-Harabasz Index: {chi_km_mm:.1f}  (↑ melhor)')

# 6. Mostrar tamanho dos clusters
uniq_mm, cnts_mm = np.unique(labels_km_mm_full, return_counts=True)
print('\nTamanho dos clusters (MinMaxScaler):')
for u, c in zip(uniq_mm, cnts_mm):
    print(f'  Cluster {u}: {c:>7,} ({c/len(df_num)*100:.1f}%)')

# 7. Mostrar perfil dos clusters (média de cada variável por cluster)
df_num_mm_clustered = df_num.copy() # Use the original df_num for profiling if possible
df_num_mm_clustered['Cluster_KM_MinMax'] = labels_km_mm_full

# Exclude 'Heart_Disease' and 'Cluster_KM_MinMax' from features for profiling
features_for_profiling = [col for col in df_num_mm_clustered.columns if col not in ['Heart_Disease', 'Cluster_KM_MinMax']]
cluster_profiles_mm = df_num_mm_clustered.groupby('Cluster_KM_MinMax')[features_for_profiling].mean(numeric_only=True)
print('\nPerfil dos clusters (MinMaxScaler):')
print(cluster_profiles_mm.T.round(2))

# 8. Visualizar clusters com projeção PCA 2D colorida por cluster
pca_mm_full = PCA(n_components=2, random_state=SEED)
X_pca_mm_full = pca_mm_full.fit_transform(X_minmax) # PCA on full minmax data

var_exp_mm = pca_mm_full.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 5))
# Plotting sample points for visualization consistency
scatter_mm = ax.scatter(X_pca_mm_full[sample_idx,0], X_pca_mm_full[sample_idx,1],
                     c=labels_km_mm_full[sample_idx], cmap='tab10',
                     alpha=0.3, s=6)
ax.legend(*scatter_mm.legend_elements(), title='Cluster')
ax.set_xlabel(f'PC1 ({var_exp_mm[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp_mm[1]*100:.1f}%)')
ax.set_title(f'K-Means (k={K_BEST_MM}) — Projeção PCA 2D (MinMaxScaler)')
plt.tight_layout(); plt.savefig('fig_kmeans_pca_minmax.png', dpi=120, bbox_inches='tight'); plt.show()


### 4.1.3 Impacto da Normalização no K-Means

> Adicionar blockquote

1.   Item de lista
2.   Item de lista





In [ ]:
km_mm = KMeans(n_clusters=K_BEST, random_state=SEED, n_init=10)
labels_km_mm = km_mm.fit_predict(X_minmax)

sil_mm = silhouette_score(X_minmax, labels_km_mm, sample_size=5000, random_state=SEED)
dbi_mm = davies_bouldin_score(X_minmax, labels_km_mm)

print(f'Comparação de normalização — K-Means k={K_BEST}:')
print(f"{'Normalização':<15} {'Silhouette':>12} {'Davies-Bouldin':>16}")
print(f"{'Z-score':<15} {sil_km:>12.4f} {dbi_km:>16.4f}")
print(f"{'Min-Max':<15} {sil_mm:>12.4f} {dbi_mm:>16.4f}")

# Visualização lado a lado
pca_mm = PCA(n_components=2, random_state=SEED)
X_mm_pca = pca_mm.fit_transform(X_minmax)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pca_pts, lbs, title in [
    (axes[0], X_pca_sample,             lbs_sample,             f'Z-score (Sil={sil_km:.3f})'),
    (axes[1], X_mm_pca[sample_idx],     labels_km_mm[sample_idx], f'Min-Max (Sil={sil_mm:.3f})'),
]:
    sc = ax.scatter(pca_pts[:,0], pca_pts[:,1], c=lbs, cmap='tab10', alpha=0.4, s=8)
    ax.set_title(f'K-Means k={K_BEST} — {title}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Impacto da Normalização nos Clusters K-Means', fontsize=13)
plt.tight_layout(); plt.savefig('fig_kmeans_norm.png', dpi=120, bbox_inches='tight'); plt.show()

## 4.2 Clustering Hierárquico (Ward)

### 4.2.1 Dendrograma

In [ ]:
N_HIER = 2000
hier_idx = np.random.choice(len(X_zscore), size=N_HIER, replace=False)
X_hier   = X_zscore.iloc[hier_idx].values

Z_linkage = linkage(X_hier, method='ward', metric='euclidean')

# Determinar threshold pelo maior salto de distância
last_merges = Z_linkage[-20:, 2]
acceleration = np.diff(last_merges, 2)
k_suggested  = acceleration[::-1].argmax() + 2
threshold    = Z_linkage[-(k_suggested-1), 2]

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z_linkage, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=9,
           color_threshold=threshold)
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Corte sugerido (d={threshold:.1f}) → {k_suggested} clusters')
ax.set_title('Dendrograma — Clustering Hierárquico (ligação Ward, amostra 2000 pts.)', fontsize=12)
ax.set_xlabel('Amostras / Tamanho do Cluster')
ax.set_ylabel('Distância (Ward)')
ax.legend()
plt.tight_layout(); plt.savefig('fig_dendrograma.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'Threshold de corte: {threshold:.2f}  →  {k_suggested} clusters sugeridos')

*Texto em itálico*### 4.2.2 Avaliação do Clustering Hierárquico

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

try:
    # 1. Re-derive df_num to ensure self-containment
    df_num_local = dataset_categorico.copy()
    df_num_local['General_Health'] = df_num_local['General_Health'].map({'Poor': 1, 'Fair': 2, 'Good': 3, 'Very Good': 4, 'Excellent': 5})
    df_num_local['Checkup'] = df_num_local['Checkup'].map({'Never': 0, '5 or more years ago': 1, 'Within the past 5 years': 2, 'Within the past 2 years': 3, 'Within the past year': 4})
    age_order = sorted(dataset_categorico['Age_Category'].unique())
    df_num_local['Age_Category'] = df_num_local['Age_Category'].map({cat: i for i, cat in enumerate(age_order)})
    df_num_local['BMI'] = df_num_local['BMI'].map({'Underweight': 1, 'Normal weight': 2, 'Overweight': 3, 'Obese': 4})
    labels_consumo = {'Low': 1, 'Medium': 2, 'High': 3}
    for col in ['Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption']:
        df_num_local[col] = df_num_local[col].map(labels_consumo)
    df_num_local['Sex'] = df_num_local['Sex'].map({'Male': 1, 'Female': 0})
    binary_cols = ['Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Smoking_History']
    for col in binary_cols:
        df_num_local[col] = df_num_local[col].map({'Yes': 1, 'No': 0})

    # 2. Data Preparation
    features = [c for c in df_num_local.columns if c != 'Heart_Disease']
    X = df_num_local[features].copy()
    scaler_zs = StandardScaler()
    X_zscore_local = pd.DataFrame(scaler_zs.fit_transform(X), columns=features)

    # 3. Sampling and Linkage
    N_HIER = 2000
    hier_idx = np.random.choice(len(X_zscore_local), size=N_HIER, replace=False)
    X_hier = X_zscore_local.iloc[hier_idx].values
    Z_linkage = linkage(X_hier, method='ward', metric='euclidean')

    # 4. Clustering & Evaluation
    K_BEST = 3
    labels_hier = fcluster(Z_linkage, t=K_BEST, criterion='maxclust') - 1
    sil_hier = silhouette_score(X_hier, labels_hier, sample_size=min(1500, N_HIER), random_state=SEED)
    dbi_hier = davies_bouldin_score(X_hier, labels_hier)

    print(f'Clustering Hierárquico ({K_BEST} clusters, Ward):')
    print(f'  Silhouette Score:     {sil_hier:.4f}')
    print(f'  Davies-Bouldin Index: {dbi_hier:.4f}')

    # 5. Visualization
    pca_h = PCA(n_components=2, random_state=SEED)
    X_hier_pca = pca_h.fit_transform(X_hier)
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(X_hier_pca[:,0], X_hier_pca[:,1], c=labels_hier, cmap='tab10', alpha=0.5, s=18)
    ax.set_title(f'Clustering Hierárquico (k={K_BEST}) — PCA 2D')
    plt.colorbar(sc, ax=ax, label='Cluster')
    plt.show()

    # 6. Profiling (Fixing the mean aggregation error)
    df_hier_orig = df_num_local.iloc[hier_idx].copy()
    df_hier_orig['Cluster'] = labels_hier
    print("\nPerfil médio (Valores Originais):")
    # Only apply mean to numeric columns to avoid category dtype error
    print(df_hier_orig.groupby('Cluster').mean(numeric_only=True).T.round(2))

except Exception as e:
    print(f"Ocorreu um erro: {e}")

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt

# 1. Calcular linkage com os outros métodos
Z_complete = linkage(X_hier, method='complete', metric='euclidean')
Z_average = linkage(X_hier, method='average', metric='euclidean')

# 2. Plotar os 3 dendrogramas lado a lado
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

linkages = [Z_linkage, Z_complete, Z_average]
names = ['Ward', 'Complete', 'Average']

for ax, Z, name in zip(axes, linkages, names):
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90)
    ax.set_title(f'Dendrograma - Método {name}')
    ax.set_ylabel('Distância')

plt.tight_layout()
plt.show()

# 3 & 4. Cortar com k=3 e calcular métricas
metodos_hier = []
for Z, name in zip(linkages, names):
    labels = fcluster(Z, t=3, criterion='maxclust')

    # Silhouette e Davies-Bouldin
    sil = silhouette_score(X_hier, labels)
    dbi = davies_bouldin_score(X_hier, labels)

    metodos_hier.append({
        'Método Linkage': name,
        'Silhouette ↑': round(sil, 4),
        'Davies-Bouldin ↓': round(dbi, 4)
    })

# 5. Mostrar tabela comparativa
df_comp_hier = pd.DataFrame(metodos_hier)
print("Comparação de Métodos Hierárquicos (k=3):")
display(df_comp_hier)

## 4.3 DBSCAN

### 4.3.1 Escolha de eps — K-Distance Graph

In [ ]:
N_DB    = 5000
db_idx  = np.random.choice(len(X_zscore), size=N_DB, replace=False)
X_db    = X_zscore.iloc[db_idx].values

MIN_PTS = 2 * X_zscore.shape[1]  # heurística: 2 × nº dimensões
print(f'MinPts = {MIN_PTS} (2 × {X_zscore.shape[1]} dimensões)')

nbrs = NearestNeighbors(n_neighbors=MIN_PTS).fit(X_db)
dists, _ = nbrs.kneighbors(X_db)
k_dists  = np.sort(dists[:, -1])

eps_base = float(np.percentile(k_dists, 92))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_dists, color='steelblue', linewidth=1.5)
ax.axhline(y=eps_base, color='red', linestyle='--', linewidth=1.5,
           label=f'eps sugerido ≈ {eps_base:.2f}')
ax.set_title(f'K-distance Graph (k={MIN_PTS}) — Escolha de eps')
ax.set_xlabel('Pontos ordenados'); ax.set_ylabel(f'Distância ao {MIN_PTS}º vizinho')
ax.legend()
plt.tight_layout(); plt.savefig('fig_dbscan_eps.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'eps sugerido: {eps_base:.2f}')

### 4.3.2 Exploração de Parâmetros DBSCAN

In [ ]:
configs = [
    {'eps': eps_base*0.7, 'min_samples': MIN_PTS},
    {'eps': eps_base,     'min_samples': MIN_PTS},
    {'eps': eps_base*1.3, 'min_samples': MIN_PTS},
    {'eps': eps_base,     'min_samples': 5},
    {'eps': eps_base,     'min_samples': MIN_PTS*2},
]

print(f"{'eps':>8} {'min_s':>7} {'clusters':>9} {'ruído%':>8} {'silhouette':>12}")
print('-' * 48)

best_db_cfg  = None
best_db_sil  = -1
best_db_lbs  = None

for cfg in configs:
    db   = DBSCAN(eps=cfg['eps'], min_samples=cfg['min_samples'], n_jobs=-1)
    lbs  = db.fit_predict(X_db)
    n_cl = len(set(lbs)) - (1 if -1 in lbs else 0)
    nois = (lbs == -1).sum() / len(lbs) * 100
    if n_cl >= 2:
        mask = lbs != -1
        sil  = silhouette_score(X_db[mask], lbs[mask],
                                sample_size=min(2000, mask.sum()), random_state=SEED)
        if sil > best_db_sil:
            best_db_sil = sil; best_db_cfg = cfg.copy(); best_db_lbs = lbs
    else:
        sil = float('nan')
    print(f"{cfg['eps']:>8.2f} {cfg['min_samples']:>7} {n_cl:>9} {nois:>7.1f}% {sil:>12.4f}")

print(f'\nMelhor configuração: eps={best_db_cfg["eps"]:.2f}, min_samples={best_db_cfg["min_samples"]}')

In [ ]:
pca_db = PCA(n_components=2, random_state=SEED)
X_db_pca = pca_db.fit_transform(X_db)

noise_mask = best_db_lbs == -1
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_db_pca[noise_mask,0], X_db_pca[noise_mask,1],
           c='lightgray', s=6, alpha=0.3, label=f'Ruído ({noise_mask.sum()/N_DB*100:.1f}%)')
sc = ax.scatter(X_db_pca[~noise_mask,0], X_db_pca[~noise_mask,1],
                c=best_db_lbs[~noise_mask], cmap='tab10', s=10, alpha=0.5)
ax.set_title(f'DBSCAN (eps={best_db_cfg["eps"]:.2f}, min_samples={best_db_cfg["min_samples"]}) — PCA 2D')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.savefig('fig_dbscan_pca.png', dpi=120, bbox_inches='tight'); plt.show()

# Perfil dos clusters DBSCAN
df_db = df_num.iloc[db_idx].copy()
df_db = df_db.apply(pd.to_numeric, errors='coerce')
df_db['Cluster'] = best_db_lbs

print('Perfil dos clusters DBSCAN (excluindo ruído):')
print(df_db[df_db['Cluster'] != -1].groupby('Cluster').mean().T.round(2))

print(f'\nTamanho dos clusters:')
for c in sorted(df_db['Cluster'].unique()):
    n = (df_db['Cluster'] == c).sum()
    label = 'Ruído' if c == -1 else f'Cluster {c}'
    print(f'  {label}: {n} ({n/len(df_db)*100:.1f}%)')

**Ponto 1 e 2**

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# Garantir que df_num tem os labels do K-Means e converter para numérico para evitar erros de dtypes
df_stats = df_num.copy().apply(pd.to_numeric)
df_stats['Cluster'] = labels_km

# --- PONTO 1: Estatísticas descritivas por cluster ---
binary_cols = ['Heart_Disease', 'Exercise', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Smoking_History', 'Sex']

stats_list = []
for cluster_id in sorted(df_stats['Cluster'].unique()):
    cluster_data = df_stats[df_stats['Cluster'] == cluster_id]

    # Estatísticas básicas para todas as colunas (exceto a de cluster)
    # Usamos numeric_only=True no agg para robustez extra
    desc = cluster_data.drop(columns=['Cluster']).agg(['mean', 'median', 'std']).T
    desc.columns = [f'C{cluster_id}_Média', f'C{cluster_id}_Mediana', f'C{cluster_id}_DesvioPadrão']

    # Percentagem para binárias (média de 0/1 é a percentagem)
    for col in binary_cols:
        desc.loc[col, f'C{cluster_id}_%_Yes'] = (cluster_data[col].mean() * 100).round(2)

    stats_list.append(desc)

# Concatenar resultados numa tabela organizada
tabela_estatisticas = pd.concat(stats_list, axis=1)
print("--- PONTO 1: Estatísticas Descritivas por Cluster ---")
display(tabela_estatisticas.round(2))

# --- PONTO 2: Exemplos representativos ---
print("\n--- PONTO 2: Exemplos Representativos (3 mais próximos do centróide) ---")

centroids = km_final.cluster_centers_  # Centróides no espaço Z-score
representative_samples = []

for i in range(km_final.n_clusters):
    # Pontos do cluster no espaço Z-score
    idx = np.where(labels_km == i)[0]
    points = X_zscore.iloc[idx].values

    # Calcular distâncias euclidianas de cada ponto ao seu centróide
    distances = cdist(points, centroids[i].reshape(1, -1), metric='euclidean').flatten()

    # Encontrar os índices das 3 menores distâncias
    closest_indices_within_cluster = np.argsort(distances)[:3]
    original_indices = df_num.index[idx[closest_indices_within_cluster]]

    # Extrair dados originais (df_num)
    samples = df_num.loc[original_indices].copy()
    samples['Cluster_ID'] = i
    samples['Dist_Centroid'] = distances[closest_indices_within_cluster]
    representative_samples.append(samples)

# Mostrar exemplos numa tabela legível
df_representative = pd.concat(representative_samples)
display(df_representative)